# SQA Benchmark — Laptop side

Run this notebook **on your laptop** (no FPGA needed).

It sweeps the same problem sizes and annealing schedule as
`SQA-Benchmark-FPGA.ipynb` using the pure-Python `SQASimulator`,
and saves the results to `laptop_results.csv`.

**After running:** open `SQA-Benchmark-Compare.ipynb` with both
`fpga_results.csv` and `laptop_results.csv` in the same folder.

In [8]:
import sys, os
# Adjust path if sqa_solver.py is in a different location relative to this notebook.
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), 'src', 'host'))

import numpy as np
import pandas as pd
from sqa_solver import SQASimulator

In [9]:
# -- Benchmark configuration -------------------------------------------------
# Must match exactly what is used in SQA-Benchmark-FPGA.ipynb.
SIZES       = [8, 16, 32, 64, 128, 256, 512, 1024]
ITERS       = 200
BETA_START  = 1 / 4096
BETA_END    = 8.0
GAMMA_START = 8.0
SEED        = 0

OUTPUT_CSV  = 'laptop_results.csv'

# Expect n=512 to take ~15 s and n=1024 ~60 s on a modern laptop.
# Reduce ITERS or remove the last entries from SIZES if you are in a hurry.


In [10]:
def make_qubo(n, seed):
    """Deterministic random dense symmetric QUBO — must be identical in both notebooks."""
    rng = np.random.default_rng(seed)
    Q = rng.uniform(-1, 1, size=(n, n))
    Q = (Q + Q.T) / 2
    np.fill_diagonal(Q, rng.uniform(-2, 0, size=n))
    return Q.astype(np.float32)


schedule = dict(iters=ITERS, beta_start=BETA_START,
                beta_end=BETA_END, gamma_start=GAMMA_START)

rows = []

for n in SIZES:
    Q = make_qubo(n, SEED)
    print(f'n={n:4d} ... ', end='', flush=True)

    sim = SQASimulator(num_trotters=8, seed=SEED)
    r = sim.solve(Q, **schedule)

    rows.append(dict(
        n             = n,
        time_s        = r.timing_s,
        best_energy   = r.best_energy,
        trotter_mean  = float(np.mean(r.all_energies)),
        trotter_std   = float(np.std(r.all_energies)),
    ))
    print(f'done  {r.timing_s:.4f} s   energy {r.best_energy:.4f}')

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)
print(f'\nSaved {OUTPUT_CSV}')
display(df)

n=   8 ... done  0.3050 s   energy -11.4172
n=  16 ... done  0.3009 s   energy -17.8760
n=  32 ... done  0.7639 s   energy -56.6299
n=  64 ... done  1.1755 s   energy -175.6495
n= 128 ... done  2.2235 s   energy -396.5571
n= 256 ... done  4.4250 s   energy -1046.3079
n= 512 ... done  9.5795 s   energy -3129.0207
n=1024 ... done  21.8246 s   energy -7951.6253

Saved laptop_results.csv


,n,time_s,best_energy,trotter_mean,trotter_std
0,8,0.305038,-11.417170,-5.894360,2.149257
1,16,0.300926,-17.876011,-9.609454,2.258012
2,32,0.763915,-56.629884,-36.312133,6.173964
3,64,1.175496,-175.649511,-111.407659,8.794975
4,128,2.223537,-396.557082,-260.324076,23.772443
5,256,4.425018,-1046.307932,-856.930790,34.143837
6,512,9.579487,-3129.020712,-2801.618638,28.872553
7,1024,21.824615,-7951.625263,-7458.564332,66.614641


---
## 2a — Iteration sweep (restarts=1)

Same configuration as `SQA-Benchmark-FPGA.ipynb` section 2a for direct comparison.  
The laptop simulator (Gibbs sequential update) should show monotonically improving  
or flat quality as iterations increase — no oscillation.

In [11]:
ITER_SWEEP_N    = 64
ITER_SWEEP_VALS = [50, 100, 200, 500, 1000, 2000, 5000]
ITER_SWEEP_CSV  = 'laptop_iter_sweep.csv'
Q_sw = make_qubo(ITER_SWEEP_N, SEED)

iter_rows = []
print(f'Iteration sweep  n={ITER_SWEEP_N}  restarts=1')
print(f'{"iters":>6}  {"time (s)":>10}  {"ms/iter":>8}  {"best energy":>14}')
print('-' * 46)

for n_iter in ITER_SWEEP_VALS:
    sim = SQASimulator(num_trotters=8, seed=SEED)
    r   = sim.solve(Q_sw, iters=n_iter,
                    beta_start=BETA_START, beta_end=BETA_END,
                    gamma_start=GAMMA_START)
    ms_per = r.timing_s / n_iter * 1000
    iter_rows.append(dict(iters=n_iter, time_s=r.timing_s, best_energy=r.best_energy))
    print(f'{n_iter:>6}  {r.timing_s:>10.4f}  {ms_per:>8.2f}  {r.best_energy:>14.4f}')

df_iter = pd.DataFrame(iter_rows)
df_iter.to_csv(ITER_SWEEP_CSV, index=False)
print(f'\nSaved {ITER_SWEEP_CSV}')
display(df_iter)


Iteration sweep  n=64  restarts=1
 iters    time (s)   ms/iter     best energy
----------------------------------------------
    50      0.3193      6.39       -175.3142
   100      0.5102      5.10       -174.8423
   200      1.0160      5.08       -175.6495
   500      3.7471      7.49       -178.3240
  1000      6.8899      6.89       -178.1851
  2000     14.7889      7.39       -179.2276
  5000     30.3716      6.07       -178.5408

Saved laptop_iter_sweep.csv


,iters,time_s,best_energy
0,50,0.319280,-175.314189
1,100,0.510180,-174.842338
2,200,1.015951,-175.649511
3,500,3.747071,-178.323974
4,1000,6.889867,-178.185064
5,2000,14.788926,-179.227579
6,5000,30.371585,-178.540775


---
## 2b — Restarts sweep (iters=200)

The laptop simulator is not bottlenecked by the Jacobi oscillation problem,  
so restarts give diminishing returns compared to the FPGA. Still useful as a baseline.

In [12]:
RESTART_VALS      = [1, 2, 3, 5, 10]
RESTART_SWEEP_CSV = 'laptop_restart_sweep.csv'

restart_rows = []
print(f'Restarts sweep  n={ITER_SWEEP_N}  iters=200')
print(f'{"restarts":>9}  {"time (s)":>10}  {"best energy":>14}')
print('-' * 40)

for nr in RESTART_VALS:
    best_e_r = np.inf
    t0 = __import__('time').perf_counter()
    for _ in range(nr):
        sim = SQASimulator(num_trotters=8, seed=SEED + _)
        r   = sim.solve(Q_sw, iters=200,
                        beta_start=BETA_START, beta_end=BETA_END,
                        gamma_start=GAMMA_START)
        if r.best_energy < best_e_r:
            best_e_r = r.best_energy
    elapsed = __import__('time').perf_counter() - t0
    restart_rows.append(dict(restarts=nr, time_s=elapsed, best_energy=best_e_r))
    print(f'{nr:>9}  {elapsed:>10.4f}  {best_e_r:>14.4f}')

df_restart = pd.DataFrame(restart_rows)
df_restart.to_csv(RESTART_SWEEP_CSV, index=False)
print(f'\nSaved {RESTART_SWEEP_CSV}')
display(df_restart)


Restarts sweep  n=64  iters=200
 restarts    time (s)     best energy
----------------------------------------
        1      1.0477       -175.6495
        2      2.1057       -175.7457
        3      2.9429       -176.9211
        5      7.9868       -177.1429
       10     12.3171       -178.0917

Saved laptop_restart_sweep.csv


,restarts,time_s,best_energy
0,1,1.047683,-175.649511
1,2,2.105712,-175.745694
2,3,2.942876,-176.921120
3,5,7.986796,-177.142907
4,10,12.317059,-178.091674
